# Imports

## Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tqdm import tqdm
import os
import matplotlib.pyplot as plt
import seaborn as sns
import math
from torch.utils.tensorboard import SummaryWriter 
import json
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import squareform
from sklearn.manifold import spectral_embedding
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import rbf_kernel

## Import Dataset Classes

In [ ]:
from dataset_classes import ISO_NE, AT, SH_Dataset

## Import Model

In [ ]:
from models_with_temporal_graph import TR_GNN_MultiScale

## Import Training and Testing Loops

In [ ]:
from helper_functions import train_model, test_model, test_model_stepwise

# Sensitivity SH

In [ ]:
def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # --- Load dataset (unscaled) ---
    dataset = SH_Dataset(
        csv_path="data/sh/shanghai.csv", # Note: using forward slashes avoids escape character issues
        T_in=72,
        T_out=240,
        lag_hours=[1, 12, 24, 168], 
        rolling_windows=[12, 24],
    )
    
    total_len = len(dataset.df_numeric)
    train_split_idx = int(0.6 * total_len)
    val_split_idx = int(0.8 * total_len)
    
    print(f"Raw data length: {total_len}")
    print(f"Scaler 'train_size' (raw rows): {train_split_idx}")
    print(f"Scaler 'val_end' (raw rows): {val_split_idx}")

    scaler = StandardScaler()
    scaler.fit(dataset.df_numeric.iloc[:train_split_idx].values.astype(np.float32))
    
    print("\n🔍 Generating Feature Clusters...")
    dataset.apply_scaler(scaler)
    dataset.scaler = scaler  
    
    train_end = train_split_idx - dataset.T_in - dataset.T_out
    val_start = train_split_idx - dataset.T_in
    val_end = val_split_idx - dataset.T_in - dataset.T_out
    test_start = val_split_idx - dataset.T_in
    
    effective_len = len(dataset)
    train_end = min(train_end, effective_len)
    val_end = min(val_end, effective_len)

    train_idx = range(0, train_end)
    val_idx = range(val_start, val_end)
    test_idx = range(test_start, effective_len)

    print(f"Total valid samples: {effective_len}")
    print(f"Train samples: {len(train_idx)}, Val samples: {len(val_idx)}, Test samples: {len(test_idx)}")
    
    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)
    test_subset = Subset(dataset, test_idx)

    # DataLoaders can be reused since batch_size doesn't change in this sensitivity analysis
    base_batch_size = 32
    train_loader = DataLoader(train_subset, batch_size=base_batch_size, shuffle=False)
    val_loader = DataLoader(val_subset, batch_size=base_batch_size, shuffle=False)
    test_loader = DataLoader(test_subset, batch_size=base_batch_size, shuffle=False)
    print(f"\n🚚 DataLoaders ready. Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

    # --- Define Static Hyperparameters ---
    static_hparams = {
        "N": dataset.N,
        "T_in": 72,
        "T_out": 240,
        "d": 32,
        "dropout_forecast": 0.1,
        "dropout_gcn": 0.2,
        "dropout_temporal": 0.2,
        "lr": 1e-4,
        "scheduler_patience": 3,
        "batch_size": base_batch_size, 
        "epochs": 100,
        "weight_decay": 1e-3, 
    }

    # --- Define Sensitivity Analysis Grids ---
    baseline_params = {
        "kernel_size": 7,
        "GCN_Layer": 5, # Referred to as GCN_Depth in prompt
        "dilation": 3,
        "hidden_dim": 64
    }

    sensitivity_grids = {
        "kernel_size": [3, 5, 11],
        "GCN_Layer": [1, 2, 3, 7],
        "dilation": [1, 2, 5],
        "hidden_dim": [32, 128, 256]
    }

    # Build the list of experiments (Baseline + One-at-a-time variations)
    experiments = [("Baseline", baseline_params.copy())]
    
    for param_name, values in sensitivity_grids.items():
        for val in values:
            # Copy baseline and change just the one parameter
            exp_params = baseline_params.copy()
            exp_params[param_name] = val
            run_name = f"{param_name}_{val}"
            experiments.append((run_name, exp_params))

    print(f"\n📊 Starting Sensitivity Analysis for {len(experiments)} total configurations.\n")

    # --- Run Experiments ---
    for exp_name, exp_params in experiments:
        print(f"{'='*60}")
        print(f"🚀 Running Experiment: {exp_name}")
        print(f"Config: {exp_params}")
        print(f"{'='*60}")
        
        # Merge static and dynamic hparams for this specific run
        current_hparams = {**static_hparams, **exp_params}

        # Initialize Model with current experiment's parameters
        model = TR_GNN_MultiScale(
            N=current_hparams["N"],
            T_in=current_hparams["T_in"],
            T_out=current_hparams["T_out"],
            d=current_hparams["d"],
            hidden_dim=current_hparams["hidden_dim"],
            GCN_Layer=current_hparams["GCN_Layer"],
            dropout_gcn=current_hparams["dropout_gcn"],
            dropout_temporal=current_hparams["dropout_temporal"],
            kernel_size=current_hparams["kernel_size"],
            dilation=current_hparams["dilation"],
        ).to(device)
        
        # Setup specific run directory
        run_title = f"TR_GNN_SH_Sensitivity/{exp_name}"
        log_dir = f"TR_GNN_SH/{run_title}"
        writer = SummaryWriter(log_dir)
        writer.add_text("hparams", json.dumps(current_hparams, indent=2))
        
        # Train Model
        model = train_model(
            model,
            train_loader,
            val_loader,
            epochs=current_hparams["epochs"],
            lr=current_hparams["lr"],
            device=device,
            scheduler_patience=current_hparams["scheduler_patience"],
            writer=writer,
            weight_decay=current_hparams["weight_decay"],
            save_path=f"Intial_Paper/{run_title}_best_model.pth" # Ensures uniquely named model weights
        )

        # Test Model
        print(f"\n🧪 Testing model performance for {exp_name}...\n")
        preds, trues = test_model(
            dataset=dataset,
            model=model, 
            test_loader=test_loader,
            device=device,
            writer=writer
        )
        
        writer.close()
        print(f"✅ Finished Experiment: {exp_name}\n")

if __name__ == "__main__":
    main()